In [ ]:
!pip install torch transformers sentencepiece datasets sudachipy sudachidict_core pyarrow requests

In [ ]:
!git clone https://github.com/mochiOS/ime.git
%cd ime

In [ ]:
!chmod +x ./vendor/ja/download.sh
!./vendor/ja/download.sh

In [ ]:
!curl https://sh.rustup.rs -sSf | sh -s -- -y

In [ ]:
import os

os.environ["PATH"] += ":/root/.cargo/bin"

!cargo --version
!rustc --version

In [ ]:
!cargo run --release -p engine --bin mimec -- \
    --lex vendor/ja/small_lex.csv \
    --lex vendor/ja/core_lex.csv \
    --matrix vendor/ja/matrix.def \
    -o vendor/ja/ja.mime

In [ ]:
import torch
import platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
	print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from datasets import load_dataset

DATASET_NAME = "hotchpotch/fineweb-2-edu-japanese"
DATASET_CONFIG = "sample_10BT"

corpus = load_dataset(
	DATASET_NAME,
	DATASET_CONFIG,
	split="train",
	streaming=True,
)

corpus

In [ ]:
import re

SENTENCE_SPLIT = re.compile(r"(?<=[。！？!?])")
JAPANESE = re.compile(r"[\u3040-\u30ff\u3400-\u9fff]")
URL = re.compile(r"https?://|www\.", re.IGNORECASE)


def split_sentences(text: str):
	text = text.replace("\r\n", "\n").replace("\r", "\n")
	text = re.sub(r"[ \t]+", " ", text)

	for paragraph in re.split(r"\n+", text):
		paragraph = paragraph.strip()

		if not paragraph:
			continue

		for sentence in SENTENCE_SPLIT.split(paragraph):
			sentence = sentence.strip()

			if sentence:
				yield sentence


def usable_sentence(sentence: str) -> bool:
	length = len(sentence)

	if length < 5 or length > 160:
		return False

	if URL.search(sentence):
		return False

	japanese = len(JAPANESE.findall(sentence))

	if japanese < 3:
		return False

	if japanese / length < 0.5:
		return False

	return True

In [ ]:
from sudachipy import Dictionary

sudachi = Dictionary().create()


def katakana_to_hiragana(text: str) -> str:
	return "".join(
		chr(ord(ch) - 0x60)
		if "\u30a1" <= ch <= "\u30f6"
		else ch
		for ch in text
	)


def to_reading(text: str) -> str:
	reading = "".join(
		morpheme.reading_form()
		for morpheme in sudachi.tokenize(text)
	)

	return katakana_to_hiragana(reading)

In [ ]:
MAX_SENTENCES = 100_000

sentences = []

for row in corpus:
	text = row.get("text")

	if not isinstance(text, str):
		continue

	for sentence in split_sentences(text):
		if not usable_sentence(sentence):
			continue

		sentences.append(sentence)

		if len(sentences) >= MAX_SENTENCES:
			break

	if len(sentences) >= MAX_SENTENCES:
		break

print("sentences:", len(sentences))

In [ ]:
!cargo build --release --bin candidates

In [74]:
import subprocess


class CandidateEngine:
	def __init__(
		self,
		executable: str,
		dictionary: str,
		limit: int = 16,
	):
		self.process = subprocess.Popen(
			[
				executable,
				"--dictionary",
				dictionary,
				"--limit",
				str(limit),
			],
			stdin=subprocess.PIPE,
			stdout=subprocess.PIPE,
			stderr=subprocess.PIPE,
			text=True,
			encoding="utf-8",
			bufsize=1,
		)

	def candidates(
		self,
		reading: str,
	) -> list[str]:
		if self.process.poll() is not None:
			error = self.process.stderr.read()

			raise RuntimeError(
				f"candidate engine terminated:\n{error}"
			)

		self.process.stdin.write(
			reading + "\n"
		)
		self.process.stdin.flush()

		count_line = self.process.stdout.readline()

		if not count_line:
			error = self.process.stderr.read()

			raise RuntimeError(
				f"candidate engine returned no response:\n{error}"
			)

		count = int(
			count_line.strip()
		)

		candidates = []

		for _ in range(count):
			line = self.process.stdout.readline()

			if not line:
				raise RuntimeError(
					"candidate engine terminated "
					"while returning candidates"
				)

			candidates.append(
				line.rstrip("\r\n")
			)

		return candidates

	def close(self) -> None:
		if self.process.poll() is not None:
			return

		self.process.stdin.close()
		self.process.wait()

	def __enter__(self):
		return self

	def __exit__(
		self,
		exc_type,
		exc_value,
		traceback,
	):
		self.close()

In [75]:
ENGINE = "target/release/candidates"
DICTIONARY = "vendor/ja/ja.mime"
N_BEST = 16

candidate_engine = CandidateEngine(
	ENGINE,
	DICTIONARY,
	N_BEST,
)

print(
	candidate_engine.candidates(
		"きょうはいいてんきですね。"
	)
)

['今日はいい転記ですね。', '今日はいい天気ですね。', '今日はいい転機ですね。', 'きょうはいい転記ですね。', '今日はいい転期ですね。', '今日はいい転帰ですね。', '今日はいい奠基ですね。', '今日はいい点鬼ですね。', '今日はいい天機ですね。', '今日はいい恬熈ですね。', '今日はいいてんきですね。', '経はいい転記ですね。', '卿はいい転記ですね。', '教はいい転記ですね。', '今日はいい転記ですゥね。', '今日はいい転記ですねェ。']


In [76]:
from tqdm.auto import tqdm

examples = []

for sentence in tqdm(sentences):
	reading = to_reading(sentence)

	if not reading:
		continue

	examples.append({
		"reading": reading,
		"positive": sentence,
	})

print("examples:", len(examples))

  0%|          | 0/100000 [00:00<?, ?it/s]

examples: 100000


In [77]:
MAX_GROUPS = 10_000

training_groups = []

for example in tqdm(examples[:MAX_GROUPS]):
	positive = example["positive"]

	candidates = candidate_engine.candidates(
		example["reading"]
	)

	seen = {positive}
	negatives = []

	for candidate in candidates:
		if candidate in seen:
			continue

		seen.add(candidate)
		negatives.append(candidate)

	if not negatives:
		continue

	training_groups.append({
		"reading": example["reading"],
		"positive": positive,
		"negatives": negatives,
	})

print("groups:", len(training_groups))

  0%|          | 0/10000 [00:00<?, ?it/s]

groups: 9999


In [78]:
import random

SEED = 42
EVAL_RATIO = 0.1

random.Random(SEED).shuffle(training_groups)

eval_count = max(
	1,
	int(len(training_groups) * EVAL_RATIO),
)

eval_groups = training_groups[:eval_count]
train_groups = training_groups[eval_count:]

print("train:", len(train_groups))
print("eval:", len(eval_groups))

train: 9000
eval: 999


In [79]:
from torch.utils.data import Dataset, DataLoader


class RankingDataset(Dataset):
	def __init__(self, groups):
		self.groups = groups

	def __len__(self):
		return len(self.groups)

	def __getitem__(self, index):
		return self.groups[index]


def collate_groups(groups):
	readings = []
	candidates = []
	group_sizes = []

	for group in groups:
		group_candidates = [
			group["positive"],
			*group["negatives"],
		]

		group_sizes.append(len(group_candidates))

		for candidate in group_candidates:
			readings.append(group["reading"])
			candidates.append(candidate)

	return {
		"readings": readings,
		"candidates": candidates,
		"group_sizes": group_sizes,
	}

In [80]:
import torch
import torch.nn.functional as F

from transformers import (
	AutoTokenizer,
	AutoModelForSequenceClassification,
)

TEACHER_MODEL = "ku-nlp/deberta-v3-base-japanese"
MAX_LENGTH = 96

device = torch.device(
	"cuda"
	if torch.cuda.is_available()
	else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(
	TEACHER_MODEL,
	use_fast=True,
)

teacher = AutoModelForSequenceClassification.from_pretrained(
	TEACHER_MODEL,
	num_labels=1,
	ignore_mismatched_sizes=True,
).to(device)

print("device:", device)

config.json:   0%|          | 0.00/736 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 1.66MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/6.28M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: ku-nlp/deberta-v3-base-japanese
Key                                        | Status     | 
-------------------------------------------+------------+-
mask_predictions.classifier.weight         | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	t

device: cuda


In [81]:
def split_scores(scores, group_sizes):
	result = []

	offset = 0

	for size in group_sizes:
		result.append(
			scores[offset:offset + size]
		)

		offset += size

	return result


def listwise_loss(scores, group_sizes):
	losses = []

	for group_scores in split_scores(
		scores,
		group_sizes,
	):
		log_probs = F.log_softmax(
			group_scores,
			dim=0,
		)

		losses.append(
			-log_probs[0]
		)

	return torch.stack(losses).mean()

In [82]:
train_loader = DataLoader(
	RankingDataset(train_groups),
	batch_size=2,
	shuffle=True,
	collate_fn=collate_groups,
)

eval_loader = DataLoader(
	RankingDataset(eval_groups),
	batch_size=4,
	shuffle=False,
	collate_fn=collate_groups,
)

print("train batches:", len(train_loader))
print("eval batches:", len(eval_loader))

train batches: 4500
eval batches: 250


In [83]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

EPOCHS = 5
LEARNING_RATE = 2e-5
GRADIENT_ACCUMULATION = 8

optimizer = AdamW(
	teacher.parameters(),
	lr=LEARNING_RATE,
	weight_decay=0.01,
)

steps_per_epoch = (
	len(train_loader)
	+ GRADIENT_ACCUMULATION - 1
) // GRADIENT_ACCUMULATION

total_steps = steps_per_epoch * EPOCHS

scheduler = get_linear_schedule_with_warmup(
	optimizer,
	num_warmup_steps=int(total_steps * 0.08),
	num_training_steps=total_steps,
)

scaler = torch.amp.GradScaler(
	"cuda",
	enabled=device.type == "cuda",
)

In [85]:
from tqdm.auto import tqdm

USE_BF16 = (
	torch.cuda.is_available()
	and torch.cuda.is_bf16_supported()
)

teacher.train()

for epoch in range(EPOCHS):
	running_loss = 0.0

	optimizer.zero_grad(
		set_to_none=True
	)

	for step, batch in enumerate(
		tqdm(
			train_loader,
			desc=f"epoch {epoch + 1}/{EPOCHS}",
		),
		start=1,
	):
		inputs = tokenizer(
			batch["readings"],
			batch["candidates"],
			padding=True,
			truncation=True,
			max_length=MAX_LENGTH,
			return_tensors="pt",
		)

		inputs = {
			key: value.to(device)
			for key, value in inputs.items()
		}

		with torch.autocast(
			device_type=device.type,
			dtype=torch.bfloat16
			if USE_BF16
			else torch.float32,
			enabled=USE_BF16,
		):
			scores = (
				teacher(**inputs)
				.logits
				.squeeze(-1)
			)

			loss = listwise_loss(
				scores,
				batch["group_sizes"],
			)

			loss = (
				loss
				/ GRADIENT_ACCUMULATION
			)

		loss.backward()

		running_loss += (
			loss.item()
			* GRADIENT_ACCUMULATION
		)

		if (
			step % GRADIENT_ACCUMULATION == 0
			or step == len(train_loader)
		):
			torch.nn.utils.clip_grad_norm_(
				teacher.parameters(),
				1.0,
			)

			optimizer.step()

			optimizer.zero_grad(
				set_to_none=True
			)

			scheduler.step()

	print(
		f"epoch {epoch + 1}: "
		f"loss={running_loss / len(train_loader):.6f}"
	)

epoch 1/5:   0%|          | 0/4500 [00:00<?, ?it/s]

epoch 1: loss=nan


epoch 2/5:   0%|          | 0/4500 [00:00<?, ?it/s]

epoch 2: loss=nan


epoch 3/5:   0%|          | 0/4500 [00:00<?, ?it/s]

epoch 3: loss=nan


epoch 4/5:   0%|          | 0/4500 [00:00<?, ?it/s]

epoch 4: loss=nan


epoch 5/5:   0%|          | 0/4500 [00:00<?, ?it/s]

epoch 5: loss=nan
